# Fase 3 — Preparação dos Dados

**Projeto:** Mapa de Calor e Predição do Risco de Acidentes de Trânsito na Rodovia BR-230  
**Disciplina:** Ciência de Dados — PPGCC / UFMA  
**Equipe:** João Batista Santos Silva Neto, Ricardo Diniz Mendes, Winy Gabrielly Soares de Oliveira  
**Metodologia:** CRISP-DM (Fase 3: Formatar, Selecionar, Limpar, Construir e Integrar)  
**Recorte:** Série pós-pandemia **2022–2025**, rodovia **BR-230**

Este notebook transforma os achados da Fase 2 em conjuntos prontos para o mapa de calor e para a modelagem preditiva. As operações seguem as tarefas do CRISP-DM e usam, de forma explícita, padrões do *Data Wrangling with pandas Cheat Sheet* (`concat`, `query`, `rename`, `assign`, seleção por colunas, `groupby`/`agg`, `merge`).


## 1. Objetivo da fase

Criar o conjunto de dados de mineração a partir dos CSV oficiais da PRF, aplicando o plano definido na Fase 2:

1. **Formatar** leitura (`latin-1`, `;`, vírgula decimal) e tipagem temporal.
2. **Integrar** os quatro anos de `datatran` (e agregados de `acidentes` quando úteis).
3. **Selecionar** o recorte BR-230 e os atributos compatíveis com o risco contextual.
4. **Limpar** sentinelas (`0` → `NaN`), traçado composto, km do Tocantins e prevalência geográfica.
5. **Construir** a variável-alvo binária, atributos derivados e partição prospectiva treino (2022–2024) / teste (2025).

A modelagem estima $P(\text{Fatal} \mid \text{Acidente}, \text{contexto})$. Não há VMD/tráfego na base; o mapa de calor continua sendo densidade histórica de ocorrência.


## 2. Ambiente, caminhos e carga reprodutível


In [ ]:
from pathlib import Path
import io
import subprocess
import sys
import tempfile
import zipfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 140)

plt.rcParams.update({
    "figure.figsize": (11, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
    "axes.titlesize": 13,
})


def localizar_raiz() -> Path:
    aqui = Path.cwd().resolve()
    for candidato in [aqui, aqui.parent]:
        if (candidato / "bases").exists() and (candidato / "entendimento_dados").exists():
            return candidato
    return aqui


RAIZ = localizar_raiz()
BASES = RAIZ / "bases"
PREPARADAS = BASES / "preparadas"
FIGURAS = RAIZ / "preparacao_dos_dados" / "figuras"
PREPARADAS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

PRF_DOWNLOAD_LINKS = {
    "datatran2025.csv": {"id": "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR", "ano": 2025},
    "datatran2024.csv": {"id": "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO", "ano": 2024},
    "datatran2023.csv": {"id": "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq", "ano": 2023},
    "datatran2022.csv": {"id": "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-", "ano": 2022},
    "acidentes2025.csv": {"id": "1-Gp9S-ALO0D1nT8S_OKoC8xlW7BY8F82", "ano": 2025},
    "acidentes2024.csv": {"id": "14lVfqdoE2gxDliaKZu7K9Mx6847maPtl", "ano": 2024},
    "acidentes2023.csv": {"id": "1-Yk6TV00CH3PixTkKmkoUJQsNiUc5xLm", "ano": 2023},
    "acidentes2022.csv": {"id": "1sd8YIqIHapYaxQeaa-6yeWMdVLTl6hgD", "ano": 2022},
}


def bundle_autoridades_ssl() -> str | bool:
    """Junta certifi às CAs do chaveiro do macOS (proxy corporativo / Netskope)."""
    try:
        import certifi
        base = Path(certifi.where()).read_text()
        fallback = certifi.where()
    except Exception:
        return True
    extras = ""
    if sys.platform == "darwin":
        try:
            extras = subprocess.check_output(
                ["security", "find-certificate", "-a", "-p", "/Library/Keychains/System.keychain"],
                text=True,
                timeout=15,
            )
        except Exception:
            extras = ""
    if "BEGIN CERTIFICATE" not in extras:
        return fallback
    destino = Path(tempfile.gettempdir()) / "cienciadados-ca-bundle.pem"
    destino.write_text(base + "\n" + extras)
    return str(destino)


def garantir_base(nome: str, pasta: Path = BASES) -> Path:
    destino = pasta / nome
    if destino.exists():
        return destino
    info = PRF_DOWNLOAD_LINKS.get(nome)
    if not info:
        raise FileNotFoundError(f"{nome} ausente em {pasta} e sem link configurado.")
    file_id = info["id"]
    print(f"Baixando {nome} (ID: {file_id})...")
    session = requests.Session()
    session.verify = bundle_autoridades_ssl()
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = session.get(url, stream=True, timeout=180)
    for k, v in r.cookies.items():
        if k.startswith("download_warning"):
            r = session.get(f"{url}&confirm={v}", stream=True, timeout=180)
            break
    if r.status_code != 200:
        raise RuntimeError(f"Falha ao baixar {nome}: HTTP {r.status_code}")
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            for membro in zf.namelist():
                if membro.endswith(".csv"):
                    with zf.open(membro) as src, open(destino, "wb") as dst:
                        dst.write(src.read())
                    print(f"✓ {destino.name} ({destino.stat().st_size / 1024**2:.1f} MB)")
                    return destino
    except zipfile.BadZipFile:
        destino.write_bytes(r.content)
        print(f"✓ {destino.name} ({destino.stat().st_size / 1024**2:.1f} MB)")
        return destino
    raise RuntimeError(f"ZIP sem CSV para {nome}")


def ler_prf(nome: str, usecols=None, **kwargs) -> pd.DataFrame:
    """Formatação: latin-1, ';' e vírgula decimal — padrão PRF."""
    caminho = garantir_base(nome)
    return pd.read_csv(
        caminho,
        sep=";",
        decimal=",",
        encoding="latin-1",
        low_memory=False,
        usecols=usecols,
        **kwargs,
    )


print("Raiz do projeto:", RAIZ)
print("Bases:", BASES)
print("Saídas preparadas:", PREPARADAS)
print("Figuras:", FIGURAS)




## 3. Formatação dos dados

A PRF exporta CSV em `ISO-8859-1`, com `;` como separador de campos e `,` como decimal. Sem esse ajuste, latitude/longitude entram como texto e a espacialização quebra. Em seguida tipamos datas/horas e normalizamos nomes de coluna.


In [ ]:
ANOS = [2022, 2023, 2024, 2025]
EXTENSAO_DNIT = {"PB": 521.0, "TO": 144.7}


def formatar_ocorrencias(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    # Tipagem temporal
    out["data"] = pd.to_datetime(out["data_inversa"], errors="coerce")
    out["hora"] = pd.to_datetime(out["horario"], format="%H:%M:%S", errors="coerce").dt.hour
    out["ano"] = out["data"].dt.year.fillna(ano).astype("Int64")
    out["mes"] = out["data"].dt.month.astype("Int64")
    out["dia"] = out["data"].dt.day.astype("Int64")
    # Coordenadas e km numéricos
    for col in ["latitude", "longitude", "km", "br"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    for col in ["pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "feridos", "veiculos"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


amostra = formatar_ocorrencias(ler_prf("datatran2025.csv"), 2025)
print("Colunas formatadas (2025):", len(amostra.columns))
print("dtypes relevantes:")
display(amostra[["data_inversa", "data", "horario", "hora", "latitude", "longitude", "km", "br"]].dtypes.to_frame("dtype"))
display(amostra.head(3))



## 4. Integração temporal e seleção da BR-230

Integramos os quatro arquivos `datatran` com `pd.concat` (empilhar linhas) e filtramos a rodovia com `query`, no espírito do cheat sheet de pandas. A unidade de análise permanece **1 linha = 1 ocorrência**.


In [ ]:
def carregar_ocorrencias(anos=ANOS, rodovia=230) -> pd.DataFrame:
    partes = []
    for ano in anos:
        bruto = ler_prf(f"datatran{ano}.csv")
        fmt = formatar_ocorrencias(bruto, ano)
        # Seleção de linhas: BR-230
        recorte = fmt.query("br == @rodovia").copy()
        partes.append(recorte)
        print(f"{ano}: nacional={len(fmt):,} | BR-230={len(recorte):,}")
    # Integração vertical (append de linhas)
    return pd.concat(partes, ignore_index=True)


br230_bruta = carregar_ocorrencias()
print(f"\nConsolidado BR-230: {len(br230_bruta):,} acidentes | "
      f"{br230_bruta['id'].nunique():,} IDs únicos | "
      f"período {br230_bruta['data'].min().date()} → {br230_bruta['data'].max().date()}")
display(
    br230_bruta.groupby("ano")
    .agg(acidentes=("id", "size"), mortos=("mortos", "sum"), feridos=("feridos", "sum"))
    .astype({"mortos": int, "feridos": int})
)



## 5. Limpeza dos dados

Aplicamos os encaminhamentos da Fase 2: variável-alvo a partir de `classificacao_acidente`, redução do traçado composto ao primeiro valor, revisão do km do Tocantins acima do DNIT, e flag de divergência UF do município × regional. Contagens de vítimas (`mortos`, `feridos`, …) permanecem no arquivo preparado para o mapa e para auditoria, mas **não entram** no conjunto de preditores.


In [ ]:
def limpar_ocorrencias(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Variável-alvo (desfecho)
    out["fatal"] = out["classificacao_acidente"].eq("Com Vítimas Fatais")
    out["alvo_fatal"] = out["fatal"].astype(int)

    # Traçado composto: "Reta;Curva" → primeiro valor + flag
    tracado = out["tracado_via"].fillna("").astype(str)
    out["tracado_composto"] = tracado.str.contains(";", regex=False)
    out["tracado_principal"] = (
        tracado.str.split(";")
        .str[0]
        .str.strip()
        .replace({"": np.nan})
    )

    # Uso do solo: Sim=Urbano, Não=Rural
    out["area_urbana"] = out["uso_solo"].eq("Sim")
    out["area_rural"] = out["uso_solo"].eq("Não")

    # Regional × município
    # regional tipicamente "SPRF-PB"; extraímos a UF da regional
    uf_regional = out["regional"].astype(str).str.extract(r"([A-Z]{2})$")[0]
    out["uf_regional"] = uf_regional
    out["uf_diverge_regional"] = out["uf"].ne(out["uf_regional"]) & out["uf_regional"].notna()

    # Quilômetro acima do cadastro DNIT (TO)
    limite = out["uf"].map(EXTENSAO_DNIT)
    out["km_acima_dnit"] = limite.notna() & (out["km"] > limite)
    # Capagem documental: marca o excesso sem descartar o registro (Fase 4 decide imputação/exclusão)
    out["km_ajustado"] = out["km"]
    mask_to = out["uf"].eq("TO") & out["km_acima_dnit"]
    out.loc[mask_to, "km_ajustado"] = EXTENSAO_DNIT["TO"]

    # Fim de semana e período noturno (atributos derivados simples)
    out["fim_de_semana"] = out["dia_semana"].isin(["sábado", "domingo", "Sábado", "Domingo"])
    # Normaliza capitalização do dia da semana
    out["dia_semana_norm"] = out["dia_semana"].astype(str).str.lower()
    out["fim_de_semana"] = out["dia_semana_norm"].isin(["sábado", "sabado", "domingo"])

    out["periodo_noturno"] = out["fase_dia"].isin(["Plena Noite", "Anoitecer", "Amanhecer"])

    # Combinação destacada na Fase 2
    out["rural_pista_simples"] = out["area_rural"] & out["tipo_pista"].eq("Simples")
    out["fora_pb"] = out["uf"].ne("PB")

    return out


br230 = limpar_ocorrencias(br230_bruta)

resumo_limpeza = pd.DataFrame([
    {"checagem": "Acidentes", "valor": len(br230)},
    {"checagem": "Fatais (alvo=1)", "valor": int(br230["alvo_fatal"].sum())},
    {"checagem": "Proporção fatal (%)", "valor": round(100 * br230["alvo_fatal"].mean(), 2)},
    {"checagem": "Traçados compostos", "valor": int(br230["tracado_composto"].sum())},
    {"checagem": "UF diverge da regional", "valor": int(br230["uf_diverge_regional"].sum())},
    {"checagem": "km acima do DNIT (TO/PB)", "valor": int(br230["km_acima_dnit"].sum())},
    {"checagem": "Rural + pista simples", "valor": int(br230["rural_pista_simples"].sum())},
    {"checagem": "Rural + simples fora PB", "valor": int((br230["rural_pista_simples"] & br230["fora_pb"]).sum())},
])
display(resumo_limpeza)

print("\nTraçado principal (top 8):")
display(br230["tracado_principal"].value_counts().head(8).rename("acidentes").to_frame())

print("Registros com km acima do DNIT:")
display(
    br230.loc[br230["km_acima_dnit"], ["id", "uf", "km", "km_ajustado", "municipio"]]
    .sort_values("km", ascending=False)
    .head(10)
)



## 6. Integração com o arquivo por pessoa

O arquivo `acidentesYYYY.csv` tem grão **1 linha = 1 pessoa**. Agregamos por `id` do acidente (motocicleta envolvida, pedestre, idade média válida) e fazemos `merge` com a base de ocorrências — padrão *join* do cheat sheet — sem inflar a contagem de acidentes.


In [ ]:
def formatar_pessoas(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    out["ano_arquivo"] = ano
    if "pesid" in out.columns:
        out["pesid"] = pd.to_numeric(out["pesid"], errors="coerce")
        out.loc[out["pesid"] == 0, "pesid"] = np.nan
    if "idade" in out.columns:
        out["idade"] = pd.to_numeric(out["idade"], errors="coerce")
        out.loc[out["idade"] == 0, "idade"] = np.nan
        out.loc[out["idade"] > 100, "idade"] = np.nan
    out["br"] = pd.to_numeric(out["br"], errors="coerce")
    return out


def colunas_pessoas(nome: str) -> list[str]:
    """Seleciona só as colunas necessárias (cheat sheet: subset de variáveis)."""
    cab = ler_prf(nome, nrows=0)
    mapa = {c.strip().lower(): c for c in cab.columns}
    wanted = ["id", "br", "pesid", "idade", "sexo", "tipo_envolvido", "tipo_veiculo"]
    return [mapa[w] for w in wanted if w in mapa]


def carregar_pessoas(anos=ANOS, rodovia=230) -> pd.DataFrame:
    partes = []
    for ano in anos:
        nome = f"acidentes{ano}.csv"
        bruto = ler_prf(nome, usecols=colunas_pessoas(nome))
        fmt = formatar_pessoas(bruto, ano)
        partes.append(fmt.query("br == @rodovia").copy())
        print(f"pessoas {ano}: BR-230={len(partes[-1]):,}")
    return pd.concat(partes, ignore_index=True)


pessoas = carregar_pessoas()
print(f"Pessoas BR-230: {len(pessoas):,} | pesid nulo: {pessoas['pesid'].isna().sum():,}")

agg_pessoas = (
    pessoas.groupby("id", dropna=False)
    .agg(
        n_pessoas_arquivo=("id", "size"),
        tem_motocicleta=("tipo_veiculo", lambda s: int(s.astype(str).str.contains("Motocicleta", case=False, na=False).any())),
        tem_pedestre=("tipo_envolvido", lambda s: int(s.eq("Pedestre").any())),
        idade_media=("idade", "mean"),
        prop_masculino=("sexo", lambda s: float((s == "Masculino").mean()) if len(s) else np.nan),
    )
    .reset_index()
)

br230_int = br230.merge(agg_pessoas, on="id", how="left", validate="one_to_one")
print(f"Após merge: {len(br230_int):,} linhas (deve permanecer = {len(br230):,})")
print(f"Acidentes sem match de pessoa: {int(br230_int['n_pessoas_arquivo'].isna().sum())}")
display(br230_int[["n_pessoas_arquivo", "tem_motocicleta", "tem_pedestre", "idade_media"]].describe().round(2))



## 7. Seleção de atributos para modelagem

Separação em três papéis:

| Papel | Exemplos | Uso |
|---|---|---|
| **Contexto (preditores)** | data/hora, UF, km, lat/lon, pista, uso do solo, clima, fase do dia, traçado | Modelo e mapa |
| **Desfecho** | `alvo_fatal`, `classificacao_acidente`, `mortos` | Alvo / auditoria |
| **Pós-ocorrência / vazamento** | contagens de feridos usadas como entrada do classificador | **Excluídas** dos preditores |

`causa_acidente` e `tipo_acidente` ficam no arquivo completo (exploração), mas **fora** do conjunto principal de preditores, porque a Fase 2 registrou que são preenchidos após a ocorrência.


In [ ]:
COLS_ID = ["id", "ano", "data", "data_inversa", "horario"]

COLS_CONTEXTO = [
    "uf", "municipio", "km", "km_ajustado", "latitude", "longitude",
    "dia_semana_norm", "hora", "mes", "fase_dia",
    "condicao_metereologica", "tipo_pista", "tracado_principal",
    "uso_solo", "sentido_via",
    "area_urbana", "area_rural", "fim_de_semana", "periodo_noturno",
    "rural_pista_simples", "fora_pb",
    "tem_motocicleta", "tem_pedestre", "idade_media",
]

COLS_ALVO = ["alvo_fatal", "fatal", "classificacao_acidente", "mortos", "feridos"]

COLS_AUDITORIA = [
    "tracado_via", "tracado_composto", "uf_regional", "uf_diverge_regional",
    "km_acima_dnit", "n_pessoas_arquivo", "causa_acidente", "tipo_acidente",
]

# Garante existência
for c in COLS_CONTEXTO + COLS_ALVO + COLS_AUDITORIA:
    if c not in br230_int.columns:
        br230_int[c] = np.nan

modelo = br230_int.loc[:, COLS_ID + COLS_CONTEXTO + ["alvo_fatal"]].copy()

print("Dimensão do conjunto de modelagem:", modelo.shape)
print("Nulos por coluna de contexto (top):")
nulos = modelo[COLS_CONTEXTO].isna().mean().sort_values(ascending=False)
display((nulos[nulos > 0] * 100).round(2).rename("% nulos").to_frame())

# Completude operacional das coordenadas
print(
    "Coordenadas nulas:",
    int(modelo["latitude"].isna().sum() + modelo["longitude"].isna().sum()),
    "| km nulo:", int(modelo["km"].isna().sum()),
)



## 8. Construção da partição prospectiva

Treino = 2022–2024; teste = 2025. A partição é temporal (não aleatória), alinhada ao uso de 2025 como horizonte prospectivo na Fase 2. O desbalanceamento (~9,7% fatais) fica documentado para a Fase 4 (métricas e ponderação), sem SMOTE nesta etapa.


In [ ]:
treino = modelo.query("ano <= 2024").copy()
teste = modelo.query("ano == 2025").copy()

particao = pd.DataFrame([
    {
        "conjunto": "Treino (2022–2024)",
        "n": len(treino),
        "fatais": int(treino["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * treino["alvo_fatal"].mean(), 2),
    },
    {
        "conjunto": "Teste (2025)",
        "n": len(teste),
        "fatais": int(teste["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * teste["alvo_fatal"].mean(), 2),
    },
    {
        "conjunto": "Total BR-230",
        "n": len(modelo),
        "fatais": int(modelo["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * modelo["alvo_fatal"].mean(), 2),
    },
])
display(particao)

# Cruzamento rural × pista (confirmação Fase 2)
cruz = (
    br230_int.assign(
        uso=np.where(br230_int["area_urbana"], "Urbano", "Rural"),
    )
    .groupby(["fora_pb", "uso", "tipo_pista"], dropna=False)
    .agg(acidentes=("id", "size"), prop_fatal=("alvo_fatal", "mean"), mortos=("mortos", "sum"))
    .reset_index()
)
cruz["recorte"] = np.where(cruz["fora_pb"], "Fora da PB", "PB")
display(
    cruz.query("uso == 'Rural' and tipo_pista == 'Simples'")
    .round({"prop_fatal": 3})
)



## 9. Visualizações de controle da preparação


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Volume e letalidade por ano
por_ano = (
    br230_int.groupby("ano")
    .agg(acidentes=("id", "size"), prop_fatal=("alvo_fatal", "mean"))
    .reset_index()
)
axes[0].bar(por_ano["ano"].astype(str), por_ano["acidentes"], color="#2c5f8a", label="Acidentes")
axes[0].set_ylabel("Acidentes")
axes[0].set_title("Volume anual — BR-230 preparada")
ax2 = axes[0].twinx()
ax2.plot(por_ano["ano"].astype(str), 100 * por_ano["prop_fatal"], color="#b42828", marker="o")
ax2.set_ylabel("Proporção fatal (%)")

# Preditores categóricos: pista × alvo
resumo_pista = (
    br230_int.groupby("tipo_pista")["alvo_fatal"]
    .agg(acidentes="size", prop="mean")
    .reindex(["Simples", "Dupla", "Múltipla"])
)
axes[1].bar(resumo_pista.index.astype(str), 100 * resumo_pista["prop"], color=["#b42828", "#2c5f8a", "#7a8699"])
axes[1].set_ylabel("Proporção fatal (%)")
axes[1].set_title("Letalidade por tipo de pista")

fig.tight_layout()
fig.savefig(FIGURAS / "preparacao_controle_anual_pista.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva em", FIGURAS / "preparacao_controle_anual_pista.png")



## 10. Persistência dos artefatos

Gravamos três CSV em `bases/preparadas/`:

- `br230_ocorrencias_preparadas.csv` — base completa limpa (mapa + auditoria + modelo)
- `br230_modelo_treino.csv` / `br230_modelo_teste.csv` — partição prospectiva com preditores contextuais + alvo


In [ ]:
# Arquivo completo (UTF-8, vírgula — padrão de saída do projeto)
caminho_full = PREPARADAS / "br230_ocorrencias_preparadas.csv"
br230_int.to_csv(caminho_full, index=False)
treino.to_csv(PREPARADAS / "br230_modelo_treino.csv", index=False)
teste.to_csv(PREPARADAS / "br230_modelo_teste.csv", index=False)
modelo.to_csv(PREPARADAS / "br230_modelo_completo.csv", index=False)

inventario = pd.DataFrame([
    {"arquivo": p.name, "linhas": sum(1 for _ in open(p, encoding="utf-8")) - 1,
     "tamanho_mb": round(p.stat().st_size / 1024**2, 2)}
    for p in sorted(PREPARADAS.glob("*.csv"))
])
display(inventario)
print("Artefatos em", PREPARADAS)



## 11. Síntese para a Fase 4 (Modelagem)

1. **Base de ocorrências:** 6.217 acidentes da BR-230 (2022–2025), coordenadas e km 100% preenchidos.
2. **Alvo:** `alvo_fatal` = 1 se `classificacao_acidente == "Com Vítimas Fatais"` — 603 casos (9,70%).
3. **Partição prospectiva:** treino 2022–2024 (4.472; 435 fatais; 9,73%) e teste 2025 (1.745; 168 fatais; 9,63%).
4. **Preditores contextuais:** tempo, espaço, via, clima e agregados de pessoa (moto, pedestre, idade média). `idade_media` tem 239 nulos (3,8%).
5. **Limpezas aplicadas:** 1.606 traçados compostos reduzidos ao principal; 109 divergências UF×regional sinalizadas; 1 km do TO acima do DNIT ajustado para 144,7.
6. **Exclusões do modelo:** contagens de vítimas e a própria classificação como entrada; causa/tipo ficam só no arquivo completo.
7. **Artefatos:** `bases/preparadas/br230_ocorrencias_preparadas.csv`, `br230_modelo_treino.csv`, `br230_modelo_teste.csv`, `br230_modelo_completo.csv`.

### Referências rápidas

- Chapman et al. *CRISP-DM 1.0* — tarefas de Data Preparation.
- RStudio. *Data Wrangling with pandas Cheat Sheet*.
- Encaminhamentos da Fase 2 (`entendimento_dados/main.tex`, matriz de decisões).

